# MCP Classification Against Custom O*NET Taxonomy

This notebook classifies MCP servers against a custom O*NET occupational task hierarchy through a three-stage LLM pipeline:

1. **Stage 1**: Screen for occupational relevance + select one of 11 high-level parent tasks
2. **Stage 2**: Select one mid-level task + assign workflow automation score (1-10)
3. **Stage 3**: Select O*NET task(s) + assign deployability score(s) (1-10)

The pipeline traverses the hierarchy dynamically, filtering options at each stage based on previous selections.

## 1. Imports and Setup

In [ ]:
import os
import re
import time
import asyncio
import pandas as pd
import numpy as np
from datetime import datetime
from dotenv import load_dotenv
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as async_tqdm

# Load environment variables
load_dotenv()

print("Imports complete.")

## 2. Configuration and Parameters

In [ ]:
# =============================================================================
# MODEL CONFIGURATION
# =============================================================================
MODEL_NAME = "gpt-4.1"  # Model to use for classification

# =============================================================================
# CONCURRENCY AND RETRY SETTINGS
# =============================================================================
MAX_CONCURRENT = 50      # Maximum concurrent API requests
MAX_RETRIES = 3          # Maximum retries per request
RETRY_BASE_DELAY = 1.0   # Base delay for exponential backoff (seconds)

# =============================================================================
# CHECKPOINTING
# =============================================================================
SAVE_EVERY = 100         # Save checkpoint every N completions
CHECKPOINT_PATH = "data/classification_checkpoint.csv"

# =============================================================================
# SAMPLE RUN CONFIGURATION
# =============================================================================
# Set to True to run on a sample instead of full dataset
RUN_SAMPLE = True

# Number of random rows to sample (used if SAMPLE_TITLES is empty)
SAMPLE_N = 10

# Specific MCP titles to test (takes priority over SAMPLE_N if non-empty)
# Add titles here to test specific MCPs
SAMPLE_TITLES = {
    # "GitLab",
    # "Blender",
    # "PostgreSQL",
    # "Puppeteer",
    # "Time",
}

# =============================================================================
# OUTPUT CONFIGURATION
# =============================================================================
OUTPUT_VERSION = "v1"    # Version tag for output files
OUTPUT_DIR = "data/llm_classification/"

print(f"Configuration loaded:")
print(f"  Model: {MODEL_NAME}")
print(f"  Max concurrent: {MAX_CONCURRENT}")
print(f"  Sample mode: {RUN_SAMPLE}")
if RUN_SAMPLE:
    if SAMPLE_TITLES:
        print(f"  Sample titles: {SAMPLE_TITLES}")
    else:
        print(f"  Sample N: {SAMPLE_N}")

## 3. Load Data

In [ ]:
# =============================================================================
# Load MCP Descriptions
# =============================================================================
mcp_df = pd.read_csv("mcp_desc_all_jan_22_cleaned.csv")
print(f"Loaded {len(mcp_df)} MCP servers")
print(f"Columns: {list(mcp_df.columns)}")

# =============================================================================
# Load Custom O*NET Taxonomy
# =============================================================================
taxonomy_df = pd.read_csv("final_onet_taxonomy.csv")
print(f"\nLoaded taxonomy with {len(taxonomy_df)} rows")
print(f"Columns: {list(taxonomy_df.columns)}")

In [ ]:
# =============================================================================
# Extract High-Level Parent Tasks (hardcoded list of 11)
# =============================================================================
HIGH_LEVEL_TASKS = taxonomy_df['high_level_task'].unique().tolist()

print(f"Found {len(HIGH_LEVEL_TASKS)} high-level parent tasks:")
for i, task in enumerate(HIGH_LEVEL_TASKS, 1):
    print(f"\n{i}. {task[:200]}{'...' if len(task) > 200 else ''}")

In [ ]:
# =============================================================================
# Build Taxonomy Lookup Dictionaries
# =============================================================================

# Map: high_level_task -> list of mid_level_tasks
high_to_mid = taxonomy_df.groupby('high_level_task')['mid_level_task'].apply(
    lambda x: x.unique().tolist()
).to_dict()

# Map: mid_level_task -> list of original_onet_tasks
mid_to_tasks = taxonomy_df.groupby('mid_level_task')['original_onet_task'].apply(
    lambda x: x.unique().tolist()
).to_dict()

print(f"Built lookup dictionaries:")
print(f"  High -> Mid mappings: {len(high_to_mid)}")
print(f"  Mid -> Task mappings: {len(mid_to_tasks)}")

# Sample verification
sample_high = HIGH_LEVEL_TASKS[0]
sample_mids = high_to_mid[sample_high]
print(f"\nSample: First high-level task has {len(sample_mids)} mid-level tasks")
if sample_mids:
    sample_mid = sample_mids[0]
    sample_tasks = mid_to_tasks.get(sample_mid, [])
    print(f"  First mid-level task has {len(sample_tasks)} O*NET tasks")

## 4. Define Prompt Templates

In [ ]:
# =============================================================================
# PROMPT 1: Occupational Relevance + High-Level Task Selection
# =============================================================================

PROMPT_1_TEMPLATE = """At the bottom of this prompt you will find a description and list of key features and use cases of an AI Model Context Protocol (MCP) server — a plugin-like system that allows AI assistants to access external tools, APIs, or data sources to perform real-world tasks.

You will answer two related questions about the MCP server. These questions are used together to determine whether the MCP performs occupationally relevant work and, if so, which branch of the O*NET taxonomy provided should be explored further to identify specific standardized tasks that may be automated or supported by the MCP.


Question 1: Occupational Relevance
<question> Does this MCP server perform or significantly enable an occupationally relevant activity — that is, a concrete work activity that humans are commonly paid to perform within the formal economy and that aligns with standardized job tasks as represented in O*NET? </question>

Follow these guidelines when answering Question 1:
- You MUST answer "Yes", "No", or "Not enough information".
- Occupational relevance can include activities that produce transferable professional work artifacts or have operational effects typical of formal occupations (e.g., code, data outputs, system behavior) as represented in ONET
- Choose "Not enough information" if the description is too vague, generic, or lacks sufficient detail about what the MCP actually does in practice. Do NOT treat the fact that something is an MCP server, tool, or framework by itself as evidence of occupational relevance. Base your decision on the specific functionality described.
- Tools primarily intended for entertainment or hobbyist use should be labeled "No" unless the MCP clearly performs a standardized occupational task.
- You may use general background knowledge to understand technologies and terms. However, base your classification only on capabilities explicitly described or directly implied by the MCP's exposed functionality. Do not infer undocumented capabilities, hypothetical uses, or future integrations.

If you do not select "Yes" for question 1, you can disregard Question 2. In this case, when providing a response, put "NA" for your answer for Question 2.


Question 2: Parent Task Selection

This step is used to determine which branch of the O*NET hierarchy should be explored in later stages in order to identify specific occupational tasks automated or supported by the MCP.

Below is a list of 11 high level parent task categories of the O*NET occupational task and work activities hierarchy:

<parent_tasks>
{parent_tasks}
</parent_tasks>

This is Question 2 that you will answer:
<question> What is the single task that best represents the core standardized occupational work activity that could reasonably be automated or supported by this MCP in common real-world deployments, based on the MCP's described capabilities, even if not explicitly stated?
 </question>

Follow these guidelines when answering Question 2:
- Select only ONE task.
- Interpret MCPs on what they actually do or directly enable. Treat connected tools as part of the MCP's capability only if their functionality is directly accessible via the MCP's app or API.
- Interpret tasks focusing on action words and what the corresponding real-world human work actually looks like in a work context.
- Consider who would own, operate, and be responsible for the workflow enabled by this MCP in a real job setting, and select tasks that reflect those human work activities as well as the underlying technical actions.
- You may use general background knowledge to understand technologies, terms, and ONET activity descriptions. However, base your classification only on capabilities explicitly described or directly implied by the MCP's exposed functionality and the meaning of the ONET items. Do not infer new technical capabilities or integrations beyond those described.


Answer ONLY in the following format and nothing else:

<question_1_answer>
Yes or No or Not enough information
</question_1_answer>

<question_2_answer>
One parent task from the above list or NA if the answer to question 1 is not "Yes."
</question_2_answer>


Here is the MCP server to answer these two questions about:

<mcp_description>
{mcp_description}
</mcp_description>"""

print("Prompt 1 template defined (Occupational Relevance + High-Level Task)")

In [ ]:
# =============================================================================
# PROMPT 2: Mid-Level Task Selection + Workflow Automation Score
# =============================================================================

PROMPT_2_TEMPLATE = """At the bottom of this prompt you will find a description and list of key features and use cases of an AI Model Context Protocol (MCP) server — a plugin-like system that allows AI assistants to access external tools, APIs, or data sources to perform real-world tasks.

You will answer two related questions about the MCP server. These questions are used together to (1) determine which mid-level task in the custom O*NET hierarchy should be explored in later stages to identify specific standardized tasks that may be automated or supported by the MCP, and (2) assess how central this MCP is as a driver of automation or augmentation across the parent task provided and the mid level task selected.


Question 1: Mid-Level Task Selection
Below is a list of mid-level tasks from the custom O*NET occupational task hierarchy, corresponding to the previously selected parent task:
"{prev_level_choice}"

<mid_level_tasks>
{mid_level_tasks}
</mid_level_tasks>

This is Question 1 that you will answer:
<question> 
What is the single mid level task that best represents the core standardized occupational work activity that could reasonably be automated or supported by this MCP in common real-world deployments, based on the MCP's described capabilities, even if not explicitly stated?
 </question>

Follow these guidelines when answering Question 1:
- Select only ONE mid level task.
- Interpret MCPs on what they actually do or directly enable. Treat connected tools as part of the MCP's capability only if their functionality is directly accessible via the MCP's app or API.
- Interpret tasks focusing on action words and what the corresponding real-world human work actually looks like in a work context.
- Consider who would own, operate, and be responsible for the workflow enabled by this MCP in a real job setting, and select tasks that reflect those human work activities as well as the underlying technical actions.
- You may use general background knowledge to understand technologies, terms, and ONET activity descriptions. However, base your classification only on capabilities explicitly described or directly implied by the MCP's exposed functionality and the meaning of the ONET items. Do not infer new technical capabilities or integrations beyond those described.


Question 2: Automation Centrality Rating

When answering question 2, you will refer to the provided parent task and the mid level task you will select.

This is Question 2 that you will answer:
<question> 
On a scale of 1–10, how central is this MCP as a driver of automation or augmentation within workflows across the selected parent and mid-level tasks? 
</question>

Follow these guidelines when answering Question 2:
- The rating should measure contribution to workflow automation, not full automation of the activity. Evaluate this MCP only with respect to the work activities listed above. Do not consider unrelated uses or hypothetical extensions.
- Base your answer strictly on the information provided above. Do not infer undocumented capabilities or future integrations.
- You may use general background knowledge to understand technologies, terms, and ONET activity descriptions. However, base your classification only on capabilities explicitly described or directly implied by the MCP's exposed functionality and the meaning of the ONET items. Do not infer undocumented capabilities, hypothetical uses, or future integrations.

Follow these scale descriptions when rating for Question 2:
1–2: Peripheral enabler. Barely contributes to workflows in these areas. Might provide minor utility in edge cases, but not central to automation.  
3–4: Niche contributor. Helps in specific, narrow scenarios within the workflow. Not widely applicable, but has some use cases.  
5–6: Moderate building block. Clear, practical contribution to workflows in these activities. Not the core automation driver, but a solid supporting component.  
7–8: Significant workflow driver. Central to many workflows in these areas. Frequently needed, widely applicable, high leverage.  
9–10: Core automation infrastructure. Foundational building block that enables many automation workflows. Without this, most automation in these activities would be much harder.


Answer ONLY in the following format and nothing else:

<question_1_answer>
One mid level task from the above list.
</question_1_answer>

<question_2_answer>
Integer from 1 to 10
</question_2_answer>


Here is the MCP server to answer these two questions about:

<mcp_description>
{mcp_description}
</mcp_description>"""

print("Prompt 2 template defined (Mid-Level Task + Workflow Automation)")

In [ ]:
# =============================================================================
# PROMPT 3: O*NET Task Selection + Deployability Score
# =============================================================================

PROMPT_3_TEMPLATE = """At the bottom of this prompt you will find a description and list of key features and use cases of an AI Model Context Protocol (MCP) server — a plugin-like system that allows AI assistants to access external tools, APIs, or data sources to perform real-world tasks.

You will answer two related questions about the MCP server. These questions are used together to (1) determine which standardized occupational tasks from the O*NET database best represents the core work activity automated by the MCP, and (2) assess how deployable the MCP is for automating that task in real-world settings.


Question 1: Task Selection

Below is a list of Tasks from the O*NET occupational tasks and work activities hierarchy, corresponding to the previously selected mid level task:
"{prev_level_choice}"

<tasks>
{onet_tasks}
</tasks>

This is Question 1 that you will answer:
<question>
Which Task(s) from the above list represent standardized occupational tasks that could reasonably be fully automated by this MCP server?
</question>

Follow these guidelines when answering the question:
- Select as many tasks as apply to the question, but do not add tasks if they do not clearly fit based on the above question.
- Interpret MCPs on what they actually do or directly enable. Treat connected tools as part of the MCP's capability only if their functionality is directly accessible via the MCP's app or API.
- Interpret tasks focusing on action words and what the corresponding real-world human work actually looks like in a work context.
- Consider who would own, operate, and be responsible for the workflow enabled by this MCP in a real job setting, and select tasks that reflect those human work activities as well as the underlying technical actions.
- You may use general background knowledge to understand technologies, terms, and ONET activity descriptions. However, base your classification only on capabilities explicitly described or directly implied by the MCP's exposed functionality and the meaning of the ONET items. Do not infer new technical capabilities or integrations beyond those described; however, you may include Tasks whose underlying human work commonly occurs when operating or deploying systems with the described capabilities.


Question 2: Deployability Rating

When answering Question 2, you will refer to each task selected in Question 1 and provide an answer for each task selected.

This is Question 2 that you will answer:
<question>
On a scale of 1–10, how deployable is this MCP for automating or supporting the selected Task — that is, the level of technical setup or expertise required to use the MCP to automate or support the Task in practice?
</question>

Follow these guidelines when answering Question 2:
- Assume the MCP is evaluated as-is, including any required API keys, authentication, or configuration implied by its description. 
- Do not assume additional engineering support unless stated.
- Base your answer only on the information provided above and what you can infer from it. Do not infer undocumented capabilities or future integrations.
- You may use general background knowledge to understand technologies, terms, and ONET activity descriptions. However, base your rating only on capabilities explicitly described or directly implied by the MCP's exposed functionality and the meaning of the ONET items. Do not infer undocumented capabilities, hypothetical uses, or future integrations.

Follow these scale descriptions when rating:
1–2: Engineer-dependent. Requires custom development, backend infrastructure, or substantial API integration by a software engineer.
3–4: IT / Technical setup required. Needs a technically skilled person to configure APIs, credentials, or workflows. Not usable by a typical worker alone.
5–6: Moderate setup / light technical skill. Requires basic technical literacy (following setup docs, installing plugins, managing API keys). No engineering required, but not plug-and-play.
7–8: Minimal setup / guided onboarding. Simple onboarding such as sign-in, granting permissions, or selecting options. A typical worker with basic digital literacy can deploy it.
9–10: Plug-and-play / immediate use. Zero setup beyond launching the tool. End-user can start using it immediately with no configuration or technical knowledge.


Answer ONLY in the following format and nothing else:

<question_1_answer>
Tasks selected from the above list, separated by semicolons
</question_1_answer>

<question_2_answer>
Integer from 1 to 10 for each task, separated by semicolons in the same order of the tasks they correspond to.
</question_2_answer>


Here is the MCP server to answer these two questions about:

<mcp_description>
{mcp_description}
</mcp_description>"""

print("Prompt 3 template defined (O*NET Tasks + Deployability)")

## 5. Response Parsing Functions

In [ ]:
def parse_prompt_1_response(response: str) -> dict:
    """
    Parse Stage 1 response to extract occupational relevance and high-level task.
    
    Returns:
        dict with keys: 'occupational_relevance', 'high_level_task', 'raw_response'
    """
    result = {
        'occupational_relevance': None,
        'high_level_task': None,
        'raw_response_1': response
    }
    
    # Extract Question 1 answer
    q1_match = re.search(r'<question_1_answer>\s*(.+?)\s*</question_1_answer>', response, re.DOTALL)
    if q1_match:
        answer = q1_match.group(1).strip()
        # Normalize the answer
        if 'yes' in answer.lower() and 'no' not in answer.lower():
            result['occupational_relevance'] = 'Yes'
        elif 'not enough' in answer.lower() or 'insufficient' in answer.lower():
            result['occupational_relevance'] = 'Not enough information'
        elif 'no' in answer.lower():
            result['occupational_relevance'] = 'No'
        else:
            result['occupational_relevance'] = answer
    
    # Extract Question 2 answer
    q2_match = re.search(r'<question_2_answer>\s*(.+?)\s*</question_2_answer>', response, re.DOTALL)
    if q2_match:
        answer = q2_match.group(1).strip()
        if answer.upper() != 'NA':
            result['high_level_task'] = answer
    
    return result


def parse_prompt_2_response(response: str) -> dict:
    """
    Parse Stage 2 response to extract mid-level task and workflow automation score.
    
    Returns:
        dict with keys: 'mid_level_task', 'workflow_automation', 'raw_response'
    """
    result = {
        'mid_level_task': None,
        'workflow_automation': None,
        'raw_response_2': response
    }
    
    # Extract Question 1 answer (mid-level task)
    q1_match = re.search(r'<question_1_answer>\s*(.+?)\s*</question_1_answer>', response, re.DOTALL)
    if q1_match:
        result['mid_level_task'] = q1_match.group(1).strip()
    
    # Extract Question 2 answer (workflow automation score)
    q2_match = re.search(r'<question_2_answer>\s*(.+?)\s*</question_2_answer>', response, re.DOTALL)
    if q2_match:
        try:
            score = int(q2_match.group(1).strip())
            result['workflow_automation'] = score
        except ValueError:
            # Try to extract just the number
            num_match = re.search(r'(\d+)', q2_match.group(1))
            if num_match:
                result['workflow_automation'] = int(num_match.group(1))
    
    return result


def parse_prompt_3_response(response: str) -> dict:
    """
    Parse Stage 3 response to extract O*NET tasks and deployability scores.
    
    Returns:
        dict with keys: 'onet_tasks', 'deployability', 'raw_response'
    """
    result = {
        'onet_tasks': None,
        'deployability': None,
        'raw_response_3': response
    }
    
    # Extract Question 1 answer (tasks, semicolon-separated)
    q1_match = re.search(r'<question_1_answer>\s*(.+?)\s*</question_1_answer>', response, re.DOTALL)
    if q1_match:
        result['onet_tasks'] = q1_match.group(1).strip()
    
    # Extract Question 2 answer (deployability scores, semicolon-separated)
    q2_match = re.search(r'<question_2_answer>\s*(.+?)\s*</question_2_answer>', response, re.DOTALL)
    if q2_match:
        result['deployability'] = q2_match.group(1).strip()
    
    return result


print("Response parsing functions defined.")

In [ ]:
# =============================================================================
# Task Matching Functions
# =============================================================================

def find_best_matching_high_level_task(llm_output: str, valid_tasks: list) -> str | None:
    """
    Find the best matching high-level task from valid options.
    Uses substring matching and similarity to handle slight variations.
    """
    if not llm_output or llm_output.upper() == 'NA':
        return None
    
    llm_lower = llm_output.lower().strip()
    
    # Try exact match first
    for task in valid_tasks:
        if task.lower() == llm_lower:
            return task
    
    # Try substring match (LLM output contained in task or vice versa)
    for task in valid_tasks:
        task_lower = task.lower()
        # Check if significant portion matches
        if llm_lower[:100] in task_lower or task_lower[:100] in llm_lower:
            return task
    
    # Try matching first 50 characters (tasks can be very long)
    for task in valid_tasks:
        if task.lower()[:50] == llm_lower[:50]:
            return task
    
    # If no match found, return the LLM output anyway (for debugging)
    return llm_output


def find_best_matching_mid_level_task(llm_output: str, valid_tasks: list) -> str | None:
    """
    Find the best matching mid-level task from valid options.
    """
    if not llm_output:
        return None
    
    llm_lower = llm_output.lower().strip()
    
    # Try exact match first
    for task in valid_tasks:
        if task.lower() == llm_lower:
            return task
    
    # Try substring match
    for task in valid_tasks:
        task_lower = task.lower()
        if llm_lower[:80] in task_lower or task_lower[:80] in llm_lower:
            return task
    
    # Try matching first 50 characters
    for task in valid_tasks:
        if task.lower()[:50] == llm_lower[:50]:
            return task
    
    return llm_output


print("Task matching functions defined.")

## 6. Async LLM Functions

In [ ]:
# =============================================================================
# Initialize Async OpenAI Client
# =============================================================================
async_client = AsyncOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

print("Async OpenAI client initialized.")

In [ ]:
async def call_llm_async(
    prompt: str,
    semaphore: asyncio.Semaphore,
    row_id: int = 0
) -> str | None:
    """
    Async LLM call with semaphore concurrency control and exponential backoff.
    
    Args:
        prompt: The formatted prompt to send
        semaphore: Asyncio semaphore for concurrency control
        row_id: Row identifier for logging
    
    Returns:
        LLM response string or None if all retries failed
    """
    async with semaphore:
        for attempt in range(MAX_RETRIES):
            try:
                resp = await async_client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.0  # Deterministic output
                )
                return resp.choices[0].message.content.strip()
            
            except Exception as e:
                delay = RETRY_BASE_DELAY * (2 ** attempt)
                print(f"  Row {row_id} retry {attempt+1}/{MAX_RETRIES}: {e}")
                await asyncio.sleep(delay)
        
        print(f"  Row {row_id} FAILED after {MAX_RETRIES} retries")
        return None


print("Async LLM call function defined.")

## 7. Three-Stage Classification Pipeline

In [ ]:
async def classify_single_mcp(
    row_idx: int,
    title: str,
    description: str,
    semaphore: asyncio.Semaphore
) -> dict:
    """
    Run the full three-stage classification pipeline for a single MCP.
    
    Returns:
        dict with all classification results
    """
    result = {
        'row_idx': row_idx,
        'title': title,
        'occupational_relevance': None,
        'high_level_task': None,
        'mid_level_task': None,
        'workflow_automation': None,
        'onet_tasks': None,
        'deployability': None,
        'raw_response_1': None,
        'raw_response_2': None,
        'raw_response_3': None,
        'error': None
    }
    
    try:
        # =====================================================================
        # STAGE 1: Occupational Relevance + High-Level Task
        # =====================================================================
        
        # Format parent tasks as numbered list
        parent_tasks_formatted = "\n".join(
            f"{i+1}. {task}" for i, task in enumerate(HIGH_LEVEL_TASKS)
        )
        
        prompt_1 = PROMPT_1_TEMPLATE.format(
            parent_tasks=parent_tasks_formatted,
            mcp_description=description
        )
        
        response_1 = await call_llm_async(prompt_1, semaphore, row_idx)
        if not response_1:
            result['error'] = 'Stage 1 failed'
            return result
        
        parsed_1 = parse_prompt_1_response(response_1)
        result.update(parsed_1)
        
        # Check if occupationally relevant - if not, stop here
        if result['occupational_relevance'] != 'Yes':
            return result
        
        # Match high-level task to valid options
        matched_high = find_best_matching_high_level_task(
            result['high_level_task'], HIGH_LEVEL_TASKS
        )
        result['high_level_task'] = matched_high
        
        if not matched_high or matched_high not in high_to_mid:
            result['error'] = f'Invalid high-level task: {result["high_level_task"]}'
            return result
        
        # =====================================================================
        # STAGE 2: Mid-Level Task + Workflow Automation
        # =====================================================================
        
        mid_level_options = high_to_mid[matched_high]
        mid_tasks_formatted = "\n".join(
            f"{i+1}. {task}" for i, task in enumerate(mid_level_options)
        )
        
        prompt_2 = PROMPT_2_TEMPLATE.format(
            prev_level_choice=matched_high[:200] + "..." if len(matched_high) > 200 else matched_high,
            mid_level_tasks=mid_tasks_formatted,
            mcp_description=description
        )
        
        response_2 = await call_llm_async(prompt_2, semaphore, row_idx)
        if not response_2:
            result['error'] = 'Stage 2 failed'
            return result
        
        parsed_2 = parse_prompt_2_response(response_2)
        result.update(parsed_2)
        
        # Match mid-level task to valid options
        matched_mid = find_best_matching_mid_level_task(
            result['mid_level_task'], mid_level_options
        )
        result['mid_level_task'] = matched_mid
        
        if not matched_mid or matched_mid not in mid_to_tasks:
            result['error'] = f'Invalid mid-level task: {result["mid_level_task"]}'
            return result
        
        # =====================================================================
        # STAGE 3: O*NET Tasks + Deployability
        # =====================================================================
        
        onet_task_options = mid_to_tasks[matched_mid]
        onet_tasks_formatted = "\n".join(
            f"{i+1}. {task}" for i, task in enumerate(onet_task_options)
        )
        
        prompt_3 = PROMPT_3_TEMPLATE.format(
            prev_level_choice=matched_mid[:200] + "..." if len(matched_mid) > 200 else matched_mid,
            onet_tasks=onet_tasks_formatted,
            mcp_description=description
        )
        
        response_3 = await call_llm_async(prompt_3, semaphore, row_idx)
        if not response_3:
            result['error'] = 'Stage 3 failed'
            return result
        
        parsed_3 = parse_prompt_3_response(response_3)
        result.update(parsed_3)
        
    except Exception as e:
        result['error'] = str(e)
    
    return result


print("Three-stage classification pipeline defined.")

## 8. Main Processing Functions

In [ ]:
async def run_classification(
    df: pd.DataFrame,
    description_col: str = 'text_for_llm_2',
    title_col: str = 'title'
) -> pd.DataFrame:
    """
    Run classification on all rows in the dataframe.
    
    Args:
        df: DataFrame with MCP data
        description_col: Column containing MCP descriptions
        title_col: Column containing MCP titles
    
    Returns:
        DataFrame with classification results
    """
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    
    print(f"Starting classification of {len(df)} MCPs...")
    print(f"  Max concurrent requests: {MAX_CONCURRENT}")
    start_time = time.time()
    
    # Create all tasks
    tasks = [
        classify_single_mcp(
            row_idx=idx,
            title=row[title_col],
            description=row[description_col] if pd.notna(row[description_col]) else "",
            semaphore=semaphore
        )
        for idx, row in df.iterrows()
    ]
    
    # Process with progress tracking
    results = []
    completed = 0
    
    for coro in asyncio.as_completed(tasks):
        result = await coro
        results.append(result)
        completed += 1
        
        # Progress update every 10 completions
        if completed % 10 == 0 or completed == len(tasks):
            elapsed = time.time() - start_time
            rate = completed / elapsed if elapsed > 0 else 0
            print(f"  Completed {completed}/{len(tasks)} ({rate:.1f}/sec)")
        
        # Checkpoint save
        if completed % SAVE_EVERY == 0:
            checkpoint_df = pd.DataFrame(results)
            os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
            checkpoint_df.to_csv(CHECKPOINT_PATH, index=False)
            print(f"  Checkpoint saved: {CHECKPOINT_PATH}")
    
    elapsed = time.time() - start_time
    print(f"\nClassification complete!")
    print(f"  Total time: {elapsed:.1f} seconds")
    print(f"  Average rate: {len(tasks)/elapsed:.1f} MCPs/sec")
    
    # Convert results to DataFrame and sort by original index
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('row_idx').reset_index(drop=True)
    
    return results_df


print("Main processing function defined.")

## 9. Prepare Data for Classification

In [ ]:
# =============================================================================
# Select Data to Classify (Sample or Full)
# =============================================================================

if RUN_SAMPLE:
    if SAMPLE_TITLES:
        # Filter by specific titles
        run_df = mcp_df[mcp_df['title'].isin(SAMPLE_TITLES)].copy()
        print(f"Sample mode: {len(run_df)} MCPs selected by title")
        print(f"  Titles: {list(run_df['title'])}")
    else:
        # Random sample
        run_df = mcp_df.sample(n=min(SAMPLE_N, len(mcp_df)), random_state=42).copy()
        print(f"Sample mode: {len(run_df)} random MCPs selected")
        print(f"  Titles: {list(run_df['title'])}")
else:
    run_df = mcp_df.copy()
    print(f"Full mode: {len(run_df)} MCPs to classify")

run_df = run_df.reset_index(drop=True)
print(f"\nData ready for classification.")

## 10. Run Classification

In [ ]:
# =============================================================================
# Run the Classification Pipeline
# =============================================================================

# For Jupyter notebooks, use nest_asyncio to allow nested event loops
import nest_asyncio
nest_asyncio.apply()

# Run classification
results_df = asyncio.run(run_classification(run_df))

print(f"\nResults shape: {results_df.shape}")
print(f"Columns: {list(results_df.columns)}")

## 11. Review Results

In [ ]:
# =============================================================================
# Summary Statistics
# =============================================================================

print("Classification Results Summary")
print("=" * 50)

# Occupational relevance distribution
print("\nOccupational Relevance:")
print(results_df['occupational_relevance'].value_counts())

# Count of successful classifications
relevant = results_df[results_df['occupational_relevance'] == 'Yes']
print(f"\nRelevant MCPs: {len(relevant)} / {len(results_df)}")

# High-level task distribution (for relevant MCPs)
if len(relevant) > 0:
    print("\nHigh-Level Task Distribution (top 5):")
    hl_counts = relevant['high_level_task'].value_counts().head(5)
    for task, count in hl_counts.items():
        print(f"  {count}: {task[:80]}...")

# Errors
errors = results_df[results_df['error'].notna()]
print(f"\nErrors: {len(errors)} / {len(results_df)}")
if len(errors) > 0:
    print("Error types:")
    print(errors['error'].value_counts())

In [ ]:
# =============================================================================
# View Sample Results
# =============================================================================

# Display key columns for first few results
display_cols = [
    'title', 
    'occupational_relevance', 
    'high_level_task',
    'mid_level_task',
    'workflow_automation',
    'onet_tasks',
    'deployability',
    'error'
]

# Truncate long text for display
display_df = results_df[display_cols].copy()
for col in ['high_level_task', 'mid_level_task', 'onet_tasks']:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(
            lambda x: x[:100] + '...' if isinstance(x, str) and len(x) > 100 else x
        )

display_df.head(10)

In [ ]:
# =============================================================================
# View Full Result for a Specific MCP
# =============================================================================

# Change this index to view different results
VIEW_INDEX = 0

if len(results_df) > VIEW_INDEX:
    row = results_df.iloc[VIEW_INDEX]
    print(f"Full Result for: {row['title']}")
    print("=" * 60)
    print(f"\nOccupational Relevance: {row['occupational_relevance']}")
    print(f"\nHigh-Level Task:\n{row['high_level_task']}")
    print(f"\nMid-Level Task:\n{row['mid_level_task']}")
    print(f"\nWorkflow Automation Score: {row['workflow_automation']}")
    print(f"\nO*NET Tasks:\n{row['onet_tasks']}")
    print(f"\nDeployability Scores: {row['deployability']}")
    if row['error']:
        print(f"\nError: {row['error']}")

## 12. Save Results

In [ ]:
# =============================================================================
# Merge Results with Original Data
# =============================================================================

# Merge classification results with original MCP data
output_df = run_df.merge(
    results_df.drop(columns=['title']),
    left_index=True,
    right_on='row_idx',
    how='left'
)

print(f"Merged output shape: {output_df.shape}")
print(f"Columns: {list(output_df.columns)}")

In [ ]:
# =============================================================================
# Save to CSV
# =============================================================================

# Create output directory if needed
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Generate output filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
if RUN_SAMPLE:
    output_filename = f"custom_tax_classification_sample_{OUTPUT_VERSION}_{timestamp}.csv"
else:
    output_filename = f"custom_tax_classification_full_{OUTPUT_VERSION}_{timestamp}.csv"

output_path = os.path.join(OUTPUT_DIR, output_filename)

# Save
output_df.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")

In [ ]:
# =============================================================================
# Save Clean Results (without raw responses)
# =============================================================================

# Columns to keep in clean output
clean_cols = [
    'title',
    'url',
    'text_for_llm_2',
    'occupational_relevance',
    'high_level_task',
    'mid_level_task',
    'workflow_automation',
    'onet_tasks',
    'deployability',
    'error'
]

clean_df = output_df[[c for c in clean_cols if c in output_df.columns]].copy()

# Save clean version
clean_filename = output_filename.replace('.csv', '_clean.csv')
clean_path = os.path.join(OUTPUT_DIR, clean_filename)
clean_df.to_csv(clean_path, index=False)
print(f"Clean results saved to: {clean_path}")

## 13. Workflow Automation & Deployability Analysis

In [ ]:
# =============================================================================
# Analyze Workflow Automation Scores
# =============================================================================

relevant_with_scores = results_df[
    (results_df['occupational_relevance'] == 'Yes') & 
    (results_df['workflow_automation'].notna())
].copy()

if len(relevant_with_scores) > 0:
    print("Workflow Automation Score Distribution:")
    print(f"  Count: {len(relevant_with_scores)}")
    print(f"  Mean: {relevant_with_scores['workflow_automation'].mean():.2f}")
    print(f"  Median: {relevant_with_scores['workflow_automation'].median():.1f}")
    print(f"  Min: {relevant_with_scores['workflow_automation'].min()}")
    print(f"  Max: {relevant_with_scores['workflow_automation'].max()}")
    print("\nScore distribution:")
    print(relevant_with_scores['workflow_automation'].value_counts().sort_index())
else:
    print("No relevant MCPs with workflow automation scores yet.")

In [ ]:
# =============================================================================
# Top MCPs by Workflow Automation
# =============================================================================

if len(relevant_with_scores) > 0:
    print("Top 10 MCPs by Workflow Automation Score:")
    top_workflow = relevant_with_scores.nlargest(10, 'workflow_automation')[
        ['title', 'workflow_automation', 'mid_level_task']
    ]
    for _, row in top_workflow.iterrows():
        mid_task = row['mid_level_task'][:60] + '...' if isinstance(row['mid_level_task'], str) and len(row['mid_level_task']) > 60 else row['mid_level_task']
        print(f"  {row['workflow_automation']}: {row['title']} -> {mid_task}")

---

## Notes

- **Sample Mode**: Set `RUN_SAMPLE = True` and either specify `SAMPLE_TITLES` or use `SAMPLE_N` for random sampling
- **Full Run**: Set `RUN_SAMPLE = False` to process all 8,953 MCPs
- **Checkpoints**: Results are saved every `SAVE_EVERY` completions to prevent data loss
- **Concurrency**: Adjust `MAX_CONCURRENT` based on API rate limits (default: 50)
- **Output**: Results saved to `data/llm_classification/` with timestamp